In [96]:
import joblib
import pandas as pd

model_pipeline = joblib.load("model.pkl")
log_transform = joblib.load("transformer.pkl")
cols = joblib.load("cols.pkl")

def fitness_function(candidate):
    new_ship = pd.DataFrame(
        [candidate],
        columns=cols
    )
        
    prediction = log_transform.inverse_transform(model_pipeline.predict(new_ship))

    if(prediction < 0):
        return 1e6
    return abs(0 - prediction)[0][0]

In [97]:
import numpy as np

GENERATIONS = 100
NEXT_GENERATION = 200
INITIAL_POPULATION_COUNT = 300
MUTATION_FACTOR = 0.10

np.random.seed(42)
df = pd.read_csv(r'C:\Users\saman\OneDrive\Desktop\Engineering Workbench\yacht_hydrodynamics.csv')

def create_population(size):
    population = []
    for _ in range(size):
        individual = []
        for i in range(6):
            col = df.columns[i]
            individual.append(np.random.uniform(low=df[col].min(), high=df[col].max()))
        population.append(individual)
    return population

initial_population = create_population(INITIAL_POPULATION_COUNT)

In [ ]:
feature_mins = df.min().to_numpy()
feature_maxs = df.max().to_numpy()
feature_ranges = feature_maxs - feature_mins

def mutate(child):
    child = child.copy()

    idx = np.random.randint(len(child))

    # Gaussian perturbation scaled to the feature's range
    change = np.random.normal(
        loc=0,
        scale=MUTATION_FACTOR * feature_ranges[idx]
    )

    child[idx] += change
    child[idx] = np.clip(
        child[idx],
        feature_mins[idx],
        feature_maxs[idx]
    )

    return child

In [99]:
def crossover(parents):
    children = []
    for _ in range(NEXT_GENERATION):
        parent1, parent2 = parents[np.random.randint(0, NEXT_GENERATION)], parents[np.random.randint(0, NEXT_GENERATION)]
        child = parent1[0:3] + parent2[3:]
        children.append(mutate(child))
    return children

In [100]:

current_generation = initial_population

for generation in range(GENERATIONS):
    fitness = []
    for candidate in current_generation:
        fitness.append(fitness_function(candidate))

    current_generation = [c for _, c in sorted(zip(fitness, current_generation))]
    children = crossover(current_generation[0:NEXT_GENERATION])
    current_generation = children

print(current_generation)

[[np.float64(-1.1484309256705685), np.float64(0.5413331289273148), np.float64(5.14), np.float64(5.20251439046237), np.float64(2.8204591245796817), np.float64(0.125)], [np.float64(-0.0007794351139225571), np.float64(0.5895693341376722), np.float64(5.072473602578424), np.float64(3.537113194589528), np.float64(3.6363192500875967), np.float64(0.42856201222506835)], [np.float64(-5.0), np.float64(0.5928479920113372), np.float64(5.014742443499533), np.float64(3.0605765424643074), np.float64(3.64), np.float64(0.324819243995853)], [np.float64(-2.1616049488776996), np.float64(0.5797285022498408), np.float64(5.012118812677907), np.float64(2.8408631996221296), np.float64(3.002032614793591), np.float64(0.27822163785938014)], [np.float64(-3.7388152026337664), np.float64(0.5337537898787241), np.float64(5.032343300476753), np.float64(3.9369814617431507), np.float64(2.9637423656171897), np.float64(0.20880669691262485)], [np.float64(-2.629748408775059), np.float64(0.5819601296928503), np.float64(5.03790

In [102]:
def eval_final_generation(gen):
    scores = []
    for candidate in gen:
        scores.append(fitness_function(candidate))
    return scores

min(eval_final_generation(current_generation))

np.float64(0.004057556913339644)

In [104]:
print(f"LOWEST RESISTANCE IN DATASET: {df['Residuary_Resistance_per_Unit_Weight_Displacement'].min()}")
print(f"LOWEST RESISTANCE FROM ALGORITHM: {min(eval_final_generation(current_generation))}")

LOWEST RESISTANCE IN DATASET: 0.01
LOWEST RESISTANCE FROM ALGORITHM: 0.004057556913339644
